# Connect IDE Clients to MCP Servers

Configure your IDE to connect to the MCP servers deployed on OpenShift via direct Route URLs.

| Mode | Endpoint | Auth |
|------|----------|------|
| **Direct Route** (this notebook) | `https://mcp-<name>-mcp-servers.apps.<domain>/mcp` | None |
| Via MaaS Gateway (Phase 3) | `https://maas.<domain>/mcp/<name>/mcp` | API Key |

> For production use with auth and rate limiting, see Phase 3 (`../3_maas/`).

**Supported IDEs:** Cursor, VS Code (Agent Mode), Claude Code

## 1. Discover MCP Server Routes

In [ ]:
import subprocess, json

result = subprocess.run(
    ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()

routes_result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

mcp_urls = {}
for line in routes_result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        short_name = name.replace("mcp-", "")
        mcp_urls[short_name] = f"https://{host}/mcp"

print(f"Cluster domain: {CLUSTER_DOMAIN}")
print(f"Discovered {len(mcp_urls)} MCP servers:")
print("")
for name, url in sorted(mcp_urls.items()):
    print(f"  {name:<20} {url}")

Expected output (5 servers):
```
code-sandbox         https://mcp-code-sandbox-mcp-servers.apps.<domain>/mcp
codebase-search      https://mcp-codebase-search-mcp-servers.apps.<domain>/mcp
context7             https://mcp-context7-mcp-servers.apps.<domain>/mcp
duckduckgo           https://mcp-duckduckgo-mcp-servers.apps.<domain>/mcp
repo-docs            https://mcp-repo-docs-mcp-servers.apps.<domain>/mcp
```

## 2. Cursor IDE

Create `.cursor/mcp.json` in your project root:

In [ ]:
cursor_config = {"mcpServers": {}}

for name, url in sorted(mcp_urls.items()):
    cursor_config["mcpServers"][name] = {"url": url}

print("=== .cursor/mcp.json ===")
print(json.dumps(cursor_config, indent=2))
print("\nCommit this file to your repo — all team members get the same MCP tools automatically.")

## 3. VS Code (Agent Mode)

Create `.vscode/mcp.json` in your project root (requires VS Code 1.100+):

In [ ]:
vscode_config = {"servers": {}}

for name, url in sorted(mcp_urls.items()):
    vscode_config["servers"][name] = {
        "type": "streamableHttp",
        "url": url
    }

print("=== .vscode/mcp.json ===")
print(json.dumps(vscode_config, indent=2))

## 4. Claude Code

Create `.mcp.json` in project root (or `~/.claude/mcp.json` for global):

In [ ]:
claude_config = {"mcpServers": {}}

for name, url in sorted(mcp_urls.items()):
    claude_config["mcpServers"][name] = {"type": "url", "url": url}

print("=== .mcp.json (Claude Code) ===")
print(json.dumps(claude_config, indent=2))
print("\nAlternatively, add via CLI:")
for name, url in sorted(mcp_urls.items()):
    print(f"  claude mcp add {name} --transport streamable-http --url \"{url}\"")

## 5. Team Deployment Tips

### Shared Project Config

Commit the MCP config to your project repository so all team members get the same tools:

```bash
# For VS Code teams
git add .vscode/mcp.json
git commit -m "Add shared MCP server configuration"

# For Cursor teams
git add .cursor/mcp.json
git commit -m "Add shared MCP server configuration"
```

### DNS Alias (Optional)

For cleaner URLs, create a Route with a custom hostname:

```yaml
apiVersion: route.openshift.io/v1
kind: Route
metadata:
  name: mcp-tools-custom
  namespace: mcp-servers
spec:
  host: mcp-tools.company.com
  to:
    kind: Service
    name: mcp-code-sandbox
  tls:
    termination: edge
```

## 6. Verification

In [ ]:
print("MCP Server Health Check")
print("=" * 60)

init_payload = json.dumps({
    "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": {"protocolVersion": "2025-03-26", "capabilities": {},
               "clientInfo": {"name": "healthcheck", "version": "1.0"}}
})

for name, url in sorted(mcp_urls.items()):
    r = subprocess.run(
        ["curl", "-sk", "-X", "POST",
         "-H", "Content-Type: application/json",
         "-H", "Accept: application/json, text/event-stream",
         "-d", init_payload,
         "-o", "/dev/null", "-w", "%{http_code}", "-m", "5", url],
        capture_output=True, text=True)
    code = r.stdout.strip()
    status = "PASS" if code in ["200", "405"] else f"FAIL ({code})"
    print(f"  [{status:<8}] {name:<20} {url}")

print("")
print("IDE verification:")
print("  Cursor:     Settings > MCP > verify green indicators")
print("  VS Code:    Command Palette > 'MCP: List Servers'")
print("  Claude Code: /mcp to list connected servers")

## Next Steps

- `../2_basic_run/1_ide_configuration.ipynb` — Full IDE setup including **model endpoint** configuration
- `../2_basic_run/2_run_public_coding_assistant.ipynb` — Run with all 5 tools (internet required)
- `../2_basic_run/3_run_closed_coding_assistant.ipynb` — Run with 3 local tools only (air-gapped)